# Data Loading & Reformatting

In this notebook, we'll download the Metacritic data set from Kaggle and check whether everything is represented correctly.  
We will also reformat the data frame. The original data contains ~300 variables that are a flattened representation of the system-dependent user review counts from the corresponding game. This should be reformatted to be made usable.

## Setup

In [33]:
# +++ Import necessary modules +++

from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from core.data import load_from_kaggle


In [23]:
# +++ Set globals +++
PROJECT_ROOT = Path.cwd().parent
RAW_DAT_DIR = PROJECT_ROOT / 'data' / 'raw'

## Load and save data set

In [ ]:
# +++ Get data set from kaggle website and save to raw folder +++

full_link = r"https://www.kaggle.com/datasets/zaireali/metacritic-games-scrape"
dataset_link = full_link.split("/datasets/")[-1]

destination = '../data/raw'
dataset_name = dataset_link.split('/')[1]

print(f'📦 Loading data set: {dataset_name}')
files = load_from_kaggle(dataset_link = dataset_link,
                         destination = destination)

print(f'✅ {len(files)} File(s) found:')
for i, file in enumerate(files, 1):
    print(f"   {i}. {file}")

📦 Loading data set: metacritic-games-scrape


100%|██████████| 5.04M/5.04M [00:00<00:00, 5.66MB/s]

Extracting files...
Loading dataset from C:\Users\janos\.cache\kagglehub\datasets\zaireali\metacritic-games-scrape\versions\2 to ../data/raw\metacritic-games-scrape
Moving file: C:\Users\janos\.cache\kagglehub\datasets\zaireali\metacritic-games-scrape\versions\2\dataset_metacritic_scraper_2025-02-15.csv to c:\Users\janos\Projects\StackFuel_PP\notebooks\../data/raw\metacritic-games-scrape
Files moved to '../data/raw\metacritic-games-scrape' directory.
✅ 1 File(s) found:
   1. dataset_metacritic_scraper_2025-02-15.csv


## Read data and do first inspection

In [46]:
# +++ Load data into workspace +++

df = pd.read_csv(RAW_DAT_DIR / files[0])

<positron-console-cell-46>:3: DtypeWarning: Columns (2,155,162,163,170,171,178,179,186,195,196,197,198) have mixed types. Specify dtype option on import or set low_memory=False.


In [47]:
# some meta information about the data set

print(f"\n🔢 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🔄 Duplicates: {df.duplicated().sum():,} ({df.duplicated().sum()/len(df)*100:.2f}%)")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


🔢 Shape: 13,429 rows × 308 columns
🔄 Duplicates: 0 (0.00%)
💾 Memory Usage: 83.98 MB


We already know that all variables after 'userscore' are the flattened output of the scraper that needs to be reformatted.  
So for now, we focus on the first 11 variables in our initial check

In [48]:
# +++ Check info on first 11 variables +++

main_vars = df.columns[:11]

df[main_vars].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13429 entries, 0 to 13428
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          13429 non-null  object
 1   genres/0       13429 non-null  object
 2   metascore      13429 non-null  object
 3   publisherName  13427 non-null  object
 4   publisherUrl   13427 non-null  object
 5   releaseDate    13397 non-null  object
 6   section        13429 non-null  object
 7   summary        13385 non-null  object
 8   type           13429 non-null  object
 9   url            13429 non-null  object
 10  userscore      13429 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [36]:
# +++ tackle the Dtypewarning from reading in the data

mixed_cols = [2, 155, 162, 163, 170, 171, 178, 179, 186, 195, 196, 197, 198]

display(df.iloc[:, mixed_cols].dtypes)

for col in df.columns[mixed_cols]:
    print(col)
    print(df[col].map(type).value_counts())

metascore                  float64
platformReviews/8/name      object
platformReviews/8/url       object
platformReviews/9/name      object
platformReviews/9/url       object
platformReviews/10/name     object
platformReviews/10/url      object
platformReviews/11/name     object
platformReviews/11/url      object
platforms/8                 object
platforms/9                 object
platforms/10                object
platforms/11                object
dtype: object

metascore
metascore
<class 'float'>    13429
Name: count, dtype: int64
platformReviews/8/name
platformReviews/8/name
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/8/url
platformReviews/8/url
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/9/name
platformReviews/9/name
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/9/url
platformReviews/9/url
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/10/name
platformReviews/10/name
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/10/url
platformReviews/10/url
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/11/name
platformReviews/11/name
<class 'float'>    13428
<class 'str'>          1
Name: count, dtype: int64
platformReviews/11/url
platformReviews/11/url
<class 'float'>    13428
<cla

It appears the we have variables that should be numerical, but are of dtype object because there are some string numbers mixed in.

In [42]:
# convert metascore to integer
df['metascore'] = pd.to_numeric(df['metascore'], errors = 'coerce').astype('Int32')

# convert the rest of flagged variables to string
for col in df.columns[mixed_cols[1:]]:
    df[col] = df[col].astype('string')

# convert the release date to datatime format
df['releaseDate'] = pd.to_datetime(df['releaseDate'])